# S6E10 — 05 Stack: combining the models

> **Pipeline position:** the **last** step. Run it after every new model: `02, 03, 04, 06, 07, 08→09` → **05**.

| | |
|---|---|
| **Goal** | Combine OOF predictions of all models into one prediction and measure honestly how much the combination gains over the best single model. |
| **Input** | `preds/oof_<name>.npy`, `preds/test_<name>.npy` for `cat_v1, cat_v2, xgb, lgbm_tuned_final, mlp, realmlp, realmlp_rich, tabpfn, tabpfn_raw, tabicl` (any subset; missing files are skipped) |
| **Output** | `preds/submission_stack_<N>models.csv` |
| **Runtime** | < 1 min |

**Why stack at all?** Different model families make different mistakes. If their errors are not perfectly correlated, a weighted combination beats each member. Here the GBDTs are almost identical to each other (see the correlation table in section 2), while neural nets and TabPFN are different enough to help.

**Three combiners**
1. **Mean of logits** — no fitting; the reference.
2. **Logistic regression on logits** — learns one weight per model (+ intercept). Few parameters → little room to overfit.
3. **Hill climbing** — greedy: repeatedly add (with replacement) the model that raises OOF AUC most; a model's weight = how often it was picked.

**Honest evaluation = nested CV.** The combiner is fitted on OOF predictions of 4 folds and scored on the 5th. Scoring it on the rows it was fitted on would be optimistic, and with nearly identical models the optimism is exactly the size of the gain we look for.
*Known small caveat:* base-model OOF predictions in the 4 fitting folds come from base models that saw the held-out fold's rows. This is standard stacking practice and the effect is negligible next to our noise (~0.0001).

> **Glossary:** *OOF* = prediction for a row from a model that did not train on it. *Logit* = log(p/(1−p)). *Spearman correlation* = how similarly two models *rank* the rows.

## 1. Setup and loading

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import warnings; warnings.filterwarnings('ignore')

N_FOLDS, RANDOM_STATE = 5, 42
CANDS = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDS if (p / 'y_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds'), Path('/kaggle/working')] if p.exists()), Path('preds'))
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
# Exactly the folds used to create every OOF file; the nested evaluation below relies on that.
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(np.zeros(len(y)), y))

# Registry of models: name -> file suffix. Missing files are skipped, so the notebook is simply rerun when a new model arrives.
NAMES = {'cat_v1': 'cat_v1', 'cat_v2': 'cat_v2', 'xgb': 'xgb', 'lgbm_tuned': 'lgbm_tuned_final', 'mlp': 'mlp', 'realmlp': 'realmlp', 'realmlp_rich': 'realmlp_rich', 'tabpfn': 'tabpfn_avg', 'tabpfn_raw': 'tabpfn_raw_avg', 'tabicl': 'tabicl'}   # model -> file suffix

# Put every model on the logit scale; probabilities from different models are squeezed near 0/1 and are not comparable in raw form.
def to_logit_pair(o, t):
    """OOF and test of one model on the SAME scale: probabilities become logits, logits stay as they are (decided on the OOF)."""
    if o.min() >= 0 and o.max() <= 1:
        f = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
        return f(o), f(t)
    return o, t

if (PREDS_DIR / 'oof_realmlp_avg.npy').exists() and (PREDS_DIR / 'test_realmlp_avg.npy').exists(): NAMES['realmlp'] = 'realmlp_avg'   # seed-averaged RealMLP
if (PREDS_DIR / 'oof_realmlp_rich_avg.npy').exists() and (PREDS_DIR / 'test_realmlp_rich_avg.npy').exists(): NAMES['realmlp_rich'] = 'realmlp_rich_avg'
OOF, TEST = {}, {}
for name, suf in NAMES.items():
    fo, ft = PREDS_DIR / f'oof_{suf}.npy', PREDS_DIR / f'test_{suf}.npy'
    if fo.exists() and ft.exists():
        o, t = np.load(fo).astype(float), np.load(ft).astype(float)
        assert len(o) == len(y), f'{name}: OOF length {len(o)} != {len(y)}'
        OOF[name], TEST[name] = to_logit_pair(o, t)
print('models used:', list(OOF), '| test rows:', {n: len(v) for n, v in TEST.items()})
assert len(OOF) >= 2, 'need at least two models in preds/'
M = list(OOF); O = np.column_stack([OOF[m] for m in M]); T = np.column_stack([TEST[m] for m in M])
assert len({len(v) for v in TEST.values()}) == 1 and not np.isnan(O).any() and not np.isnan(T).any()

models used: ['cat_v1', 'cat_v2', 'xgb', 'lgbm_tuned', 'mlp', 'realmlp', 'realmlp_rich', 'tabpfn', 'tabpfn_raw', 'tabicl'] | test rows: {'cat_v1': 299844, 'cat_v2': 299844, 'xgb': 299844, 'lgbm_tuned': 299844, 'mlp': 299844, 'realmlp': 299844, 'realmlp_rich': 299844, 'tabpfn': 299844, 'tabpfn_raw': 299844, 'tabicl': 299844}


## 2. Single models and how similar they are

In [2]:
auc = lambda yy, s: roc_auc_score(yy, s)
# Per-fold AUC is shown, not only the mean: it reveals whether a model is consistently better or just lucky on one fold.
single = pd.DataFrame({'model': M, 'OOF AUC': [round(auc(y, O[:, i]), 5) for i in range(len(M))],
                       'fold AUCs': [[round(auc(y[va], O[va, i]), 5) for _, va in FOLDS] for i in range(len(M))]})
print(single.to_string(index=False))
print('\nSpearman correlation of OOF predictions:')
print(pd.DataFrame(spearmanr(O)[0], index=M, columns=M).round(4).to_string())

       model  OOF AUC                                     fold AUCs
      cat_v1  0.96064   [0.96074, 0.95971, 0.9614, 0.96074, 0.9606]
      cat_v2  0.96062  [0.9606, 0.95965, 0.96138, 0.96078, 0.96075]
         xgb  0.96066  [0.96074, 0.95981, 0.96136, 0.9608, 0.96061]
  lgbm_tuned  0.96088   [0.9611, 0.96002, 0.9616, 0.96094, 0.96083]
         mlp  0.96043  [0.96018, 0.9594, 0.96128, 0.96084, 0.96048]
     realmlp  0.96082 [0.96097, 0.95997, 0.96163, 0.96087, 0.96069]
realmlp_rich  0.96129 [0.96143, 0.96035, 0.96207, 0.96138, 0.96124]
      tabpfn  0.96158 [0.96167, 0.96081, 0.96212, 0.96183, 0.96149]
  tabpfn_raw  0.96089  [0.96082, 0.9601, 0.96148, 0.96122, 0.96086]
      tabicl  0.95759  [0.9574, 0.95693, 0.95813, 0.95785, 0.95766]

Spearman correlation of OOF predictions:
              cat_v1  cat_v2     xgb  lgbm_tuned     mlp  realmlp  realmlp_rich  tabpfn  tabpfn_raw  tabicl
cat_v1        1.0000  0.9949  0.9877      0.9870  0.9769   0.9814        0.9801  0.9705      0.9602  0

## 3. Combiners and nested CV
`fit_*` take the OOF matrix and labels of the fitting rows and return a weight vector (one per model); the combined score is `logits @ w` (+ intercept, irrelevant for AUC).

In [3]:
def fit_mean(Of, yf):  return np.ones(Of.shape[1]) / Of.shape[1]

# C=10: weak regularisation. With 10 inputs and 700k rows, overfitting risk is minimal; negative coefficients are allowed (see notes below).
def fit_lr(Of, yf):
    m = LogisticRegression(C=10.0, max_iter=500).fit(Of, yf)
    return m.coef_[0]

# Hill climbing: a greedy, non-negative alternative; used as a cross-check of the LR weights.
def fit_hill(Of, yf, iters=40, sub=200_000, seed=0):
    # greedy forward selection with replacement; AUC on a fixed random subset of the fitting rows for speed
    rs = np.random.RandomState(seed); idx = rs.choice(len(yf), min(sub, len(yf)), replace=False)
    Os, ys = Of[idx], yf[idx]; counts = np.zeros(Of.shape[1]); tot = np.zeros(len(idx))
    for _ in range(iters):
        sc = [auc(ys, tot + Os[:, j]) for j in range(Of.shape[1])]      # AUC of the sum = AUC of the mean
        j = int(np.argmax(sc)); counts[j] += 1; tot += Os[:, j]
    return counts / counts.sum()

COMBINERS = {'mean of logits': fit_mean, 'logistic regression': fit_lr, 'hill climbing': fit_hill}

res = {k: [] for k in COMBINERS}; W = {k: [] for k in COMBINERS}
best_single_idx = int(np.argmax([auc(y, O[:, i]) for i in range(len(M))]))
res['best single model (' + M[best_single_idx] + ')'] = []
# NESTED evaluation: fit the combiner on 4 folds of OOF predictions, score on the 5th. Scoring on the rows used for fitting would be optimistic.
for tr, va in FOLDS:
    for name, fit in COMBINERS.items():
        w = fit(O[tr], y[tr]); W[name].append(w); res[name].append(auc(y[va], O[va] @ w))
    res['best single model (' + M[best_single_idx] + ')'].append(auc(y[va], O[va, best_single_idx]))
tab = pd.DataFrame({k: v for k, v in res.items()}, index=[f'fold {i}' for i in range(N_FOLDS)]).T
tab['mean'] = tab.mean(axis=1)
print(tab.round(5).to_string())
bs = tab.index[-1]
print('\ngain vs best single model (mean over folds) and per-fold signs:')
for k in COMBINERS:
    d = tab.loc[k].iloc[:N_FOLDS] - tab.loc[bs].iloc[:N_FOLDS]
    print(f'  {k:22s} mean {d.mean():+.5f} | per fold {np.round(d.values, 5).tolist()}')
print('\nweights (mean over outer folds; for LR: raw coefficients):')
print(pd.DataFrame({k: np.mean(W[k], axis=0) for k in COMBINERS}, index=M).round(3).to_string())

                             fold 0   fold 1   fold 2   fold 3   fold 4     mean
mean of logits              0.96148  0.96061  0.96214  0.96159  0.96141  0.96144
logistic regression         0.96191  0.96095  0.96240  0.96194  0.96170  0.96178
hill climbing               0.96186  0.96093  0.96239  0.96190  0.96168  0.96175
best single model (tabpfn)  0.96167  0.96081  0.96212  0.96183  0.96149  0.96158

gain vs best single model (mean over folds) and per-fold signs:
  mean of logits         mean -0.00014 | per fold [-0.00019, -0.0002, 2e-05, -0.00024, -8e-05]
  logistic regression    mean +0.00020 | per fold [0.00024, 0.00014, 0.00028, 0.00011, 0.00021]
  hill climbing          mean +0.00017 | per fold [0.00019, 0.00013, 0.00027, 7e-05, 0.00019]

weights (mean over outer folds; for LR: raw coefficients):
              mean of logits  logistic regression  hill climbing
cat_v1                   0.1               -0.037          0.000
cat_v2                   0.1               -0.104      

## 4. Choosing the combiner and writing the submission
Rule: use logistic regression unless hill climbing beats it by more than the fold-to-fold noise (~0.0001) in the nested table above; use the plain mean if both fitted combiners are not better than it. The simplest combiner that is not worse is preferred.
The chosen combiner is then fitted on **all** OOF rows and applied to the test logits.

In [4]:
# Decision rule is in the markdown above: simplest combiner that is not worse than the others.
CHOICE = 'logistic regression'      # change after reading the table above: 'mean of logits' | 'logistic regression' | 'hill climbing'
w = COMBINERS[CHOICE](O, y)
print('final weights:', dict(zip(M, np.round(w, 3))))
oof_stack = O @ w
print(f'in-sample OOF AUC of the final stack (optimistic, for reference only): {auc(y, oof_stack):.5f}')
print(f'nested-CV AUC of the same combiner (honest): {tab.loc[CHOICE].iloc[:N_FOLDS].mean():.5f}')

raw_dir = next((p for p in [Path('data/raw'), Path('../data/raw'), Path('/kaggle/input/playground-series-s6e10')] if (p / 'sample_submission.csv').exists()), None)
sub = pd.read_csv(raw_dir / 'sample_submission.csv')
assert len(sub) == len(T), (len(sub), len(T))
test_ids = pd.read_csv(raw_dir / 'test.csv', usecols=['id'])['id']
assert (test_ids.values == sub['id'].values).all(), 'sample_submission ids differ from test.csv order'
sub[sub.columns[1]] = 1 / (1 + np.exp(-(T @ w)))
out = PREDS_DIR / f'submission_stack_{len(M)}models.csv'; sub.to_csv(out, index=False)
print('saved', out, '| prediction range', sub.iloc[:, 1].min().round(4), sub.iloc[:, 1].max().round(4), '| mean', sub.iloc[:, 1].mean().round(4))

final weights: {'cat_v1': np.float64(-0.032), 'cat_v2': np.float64(-0.105), 'xgb': np.float64(0.026), 'lgbm_tuned': np.float64(0.178), 'mlp': np.float64(-0.012), 'realmlp': np.float64(-0.112), 'realmlp_rich': np.float64(0.387), 'tabpfn': np.float64(0.657), 'tabpfn_raw': np.float64(0.112), 'tabicl': np.float64(-0.091)}
in-sample OOF AUC of the final stack (optimistic, for reference only): 0.96178
nested-CV AUC of the same combiner (honest): 0.96178
saved preds/submission_stack_10models.csv | prediction range 0.0086 0.9979 | mean 0.4453


## 5. How to read the results
* **Gain of the best combiner over the best single model, per fold:** all positive and mean > +0.0002 means the stack is real.
* **Weights:** a model with weight ≈ 0 adds nothing beyond the others. Negative weights in logistic regression are legitimate (the model is used to *correct* correlated models) — but check that they are stable across folds.
* **Public LB is noisy (±0.0003–0.0005),** so decisions are taken on nested CV, not on the leaderboard.

## Results & decisions (10-model stack)

| Combiner | Nested CV AUC | Gain vs best single (`tabpfn` 0.96158) |
|---|---|---|
| mean of logits | 0.96144 | **−0.00014** (worse in 4 of 5 folds) |
| **logistic regression** | **0.96178** | **+0.00020**, positive on **all 5 folds** |
| hill climbing | 0.96175 | +0.00017 |

**Public LB of the submitted stack: 0.96132** (6-model 0.96118, 8-model 0.96124).

**Final LR weights:** `tabpfn` 0.66, `realmlp_rich` 0.39, `lgbm_tuned` 0.18, `tabpfn_raw` 0.11; small negatives for `cat_v2`, `realmlp`, `tabicl` (≈ −0.1).

**Why these choices**
- **Logistic regression is chosen:** best, and hill climbing is not better by more than the noise.
- **The plain mean loses to the best single model** because it gives equal weight to weak members (`tabicl` 0.95759); learning weights is what makes stacking pay off.
- **Weights show what matters:** TabPFN and the rich-feature RealMLP carry the stack; the extra GBDTs add almost nothing. New ideas should therefore target *new kinds* of strong models, not more GBDT variants.
- **Negative weights** mean "this model is correlated with a better one; subtract the shared part". They are small and stable across folds.